In [1]:
from parser import Parser
import shapely
from shapely.ops import transform
import osmnx as ox
import geopandas as gpd
import pathlib
import os


import networkx as nx
import numpy as np
from shapely.geometry import LineString, Point
import math
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt

import numpy.typing as npt

from scipy.spatial import cKDTree
from shapely.strtree import STRtree

from shapely.geometry import LineString
import math




In [2]:
save_dir = pathlib.Path("/mnt/c/Users/nikita/qgisData/busroutes")
save_dir.mkdir(parents=True, exist_ok=True)

In [3]:
with open("input/boundaries.geojson", "r") as f:
    geojson = f.read()
boundaries = shapely.from_geojson(geojson)
graph = ox.graph_from_polygon(boundaries, network_type="drive", simplify=False)
nodes, edges = ox.graph_to_gdfs(graph)


# convert crs to mercator
nodes = nodes.to_crs(epsg=3857)
edges = edges.to_crs(epsg=3857)

In [4]:
bus_parser = Parser.BusGraphParser("Санкт-Петербург")
_example_route = bus_parser.get_route("/spb/bus/61")

# convert to mercator
example_route_series = gpd.GeoSeries([Point(y, x) for x, y in _example_route], crs=4326).to_crs(epsg=3857)
example_route_series


read https://kudikina.ru/spb/bus/61/map from cache


0      POINT (3394516.106 8401817.854)
1      POINT (3394524.566 8401795.807)
2      POINT (3394527.349 8401787.567)
3      POINT (3394531.468 8401772.646)
4       POINT (3394537.256 8401754.83)
                    ...               
430    POINT (3394544.047 8401737.682)
431    POINT (3394570.096 8401771.978)
432    POINT (3394572.099 8401806.051)
433     POINT (3394563.751 8401819.19)
434     POINT (3394539.483 8401810.06)
Length: 435, dtype: geometry

In [19]:

geoms = edges["geometry"]


rtree = STRtree(edges["geometry"])





LINESTRING (3394516.1057665832 8401817.854236836, 3394524.5660478836 8401795.80678464)


array([(1539245550, 1539245573, 0), (1539245573, 1539245550, 0),
       (1539245573, 11430921155, 0), (11430921155, 1539245573, 0),
       (11430921155, 12604034793, 0), (12604034793, 11430921155, 0),
       (1539245550, 1836112211, 0), (1836112211, 1539245550, 0),
       (12604034793, 1539245589, 0), (1539245589, 12604034793, 0)],
      dtype=object)

In [31]:
def calc_angle_dot_product(a: npt.NDArray, b: npt.NDArray) -> float:
    """
    Calculate the angle between two vectors using the dot product formula.
    """
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    return np.dot(a, b)

In [41]:
def snap_point_to_best_edge(gr_edges, idxtree, route_part, distance_tol=0.0004):
    
    edge_idx = idxtree.query(route_part.buffer(distance_tol))
    edge_candidates = geoms.iloc[edge_idx].index.to_numpy()
    # Pick the best matching edge
    best_match = None
    

    for u, v, key in edge_candidates:
        edge = gr_edges.loc[(u, v, key)]
        edge_geom = edge['geometry']
        edge_geom = LineString(edge_geom)
        
        
        # calc angle bween route and edge as angle between two vectors using dot product
        dot_prod = calc_angle_dot_product(
            np.array(route_part.coords[0]) - np.array(route_part.coords[1]),
            np.array(edge_geom.coords[0]) - np.array(edge_geom.coords[1])
        )
        print(dot_prod)



        # find best dot product
        if best_match is None or dot_prod > best_match[0]:
            best_match = (dot_prod, edge)
            print("best match", best_match[0])

        
            

    return best_match

In [43]:
buf_tol = 20
ln = LineString([example_route_series.iloc[6], example_route_series.iloc[7]])
buf = ln.buffer(buf_tol)
hm = rtree.query(buf)
with open(save_dir/"linebuf.geojson", "w") as f:
    f.write(shapely.to_geojson(buf))
ne_array = geoms.iloc[hm].index.to_numpy()
ne_array
matchedges = edges.loc[ne_array]
with open(save_dir / "matchedges.geojson", "w") as f:
    f.write(matchedges.to_json())
mat = snap_point_to_best_edge(edges, rtree, LineString([example_route_series.iloc[6], example_route_series.iloc[7]]), distance_tol=buf_tol)
mat

0.9997353569782665
best match 0.9997353569782665
-0.9997353569782665
0.9999992738688882
best match 0.9999992738688882
-0.9999992738688882
-0.9995898872778174
0.9995898872778174


(np.float64(0.9999992738688882),
 osmid                                               133743584
 highway                                              tertiary
 junction                                                  NaN
 lanes                                                       2
 maxspeed                                             RU:rural
 name                                         Челябинский мост
 oneway                                                  False
 reversed                                                 True
 length                                             117.629495
 ref                                                       NaN
 bridge                                                    yes
 width                                                     NaN
 tunnel                                                    NaN
 access                                                    NaN
 geometry    LINESTRING (3394368.9302678057 8401644.7723769...
 Name: (245837, 262399

In [44]:
mat

(np.float64(0.9999992738688882),
 osmid                                               133743584
 highway                                              tertiary
 junction                                                  NaN
 lanes                                                       2
 maxspeed                                             RU:rural
 name                                         Челябинский мост
 oneway                                                  False
 reversed                                                 True
 length                                             117.629495
 ref                                                       NaN
 bridge                                                    yes
 width                                                     NaN
 tunnel                                                    NaN
 access                                                    NaN
 geometry    LINESTRING (3394368.9302678057 8401644.7723769...
 Name: (245837, 262399

In [ ]:
matched_edges = []
for i in range(len(example_route) - 1):
    pt1 = example_route[i]
    pt2 = example_route[i + 1]
    h = calculate_heading(pt1, pt2)
    match = snap_point_to_best_edge(graph, pt1[0], pt1[1], heading=h)
    if match:
        matched_edges.append(match)

In [ ]:


# Remove duplicates
clean_edges = [matched_edges[0]]
for e in matched_edges[1:]:
    if e != clean_edges[-1]:
        clean_edges.append(e)

# Build a valid path (using shortest paths if needed between edge endpoints)
final_path = []
for u, v in clean_edges:
    try:
        segment = nx.shortest_path(graph, source=u, target=v, weight='length')
        final_path.extend(segment[:-1])  # avoid duplication
    except nx.NetworkXNoPath:
        continue
if clean_edges:
    final_path.append(clean_edges[-1][1])





In [ ]:
ox.plot_graph_routes(graph, [final_path], route_color='red', route_linewidth=4)

In [ ]:
# with open(save_dir / "boundaries.geojson", "w") as f:
#     f.write(shapely.to_geojson(boundaries))

# if "edges.geojson" not in os.listdir(save_dir):
#     with open(save_dir / "edges.geojson", "w") as f:
#         f.write(edges.to_json())

# if "nodes.geojson" not in os.listdir(save_dir):
#     with open(save_dir / "nodes.geojson", "w") as f:
#         f.write(nodes.to_json())

# near_points.to_file(save_dir/"points.geojson", driver="GeoJSON")
# edges.to_file(save_dir/"edges.geojson", driver="GeoJSON")
# nodes.to_file(save_dir/"nodes.geojson", driver="GeoJSON")